In [183]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge

##EDA Section

In [184]:
X_test = pd.read_csv('test.csv')

In [185]:
df_train = pd.read_csv('train.csv')



In [186]:
drop_cols = ['id', 'building_id']
df_train = df_train.drop(drop_cols, axis=1)

In [187]:
categorical_cols = ['building_type', 'day_of_week']
df_train_hot = pd.get_dummies(df_train, columns=categorical_cols, drop_first=True)

X_test_copy = X_test.copy().drop(drop_cols, axis=1)
X_test_hot = pd.get_dummies(X_test_copy, columns=categorical_cols, drop_first=True)

In [188]:
df_train.head()

,building_type,hour,day_of_week,month,temperature,humidity,occupancy,previous_usage,energy_usage
0,LectureHall,2,Monday,9,25.2,81.9,31.0,28.0,27.2
1,Science,1,Wednesday,11,25.7,83.7,14.0,72.6,67.2
2,Residential,5,Sunday,1,26.3,84.7,199.0,51.6,54.2
3,Engineering,21,Wednesday,5,29.1,77.4,68.0,68.3,63.6
4,Sports,16,Wednesday,5,30.9,68.2,65.0,49.1,53.9


In [189]:
df_train_hot

,hour,month,temperature,humidity,occupancy,previous_usage,energy_usage,building_type_Business,building_type_Engineering,building_type_LectureHall,building_type_Library,building_type_Residential,building_type_Science,building_type_Sports,day_of_week_Monday,day_of_week_Saturday,day_of_week_Sunday,day_of_week_Thursday,day_of_week_Tuesday,day_of_week_Wednesday
0,2,9,25.2,81.9,31.0,28.0,27.2,False,False,True,False,False,False,False,True,False,False,False,False,False
1,1,11,25.7,83.7,14.0,72.6,67.2,False,False,False,False,False,True,False,False,False,False,False,False,True
2,5,1,26.3,84.7,199.0,51.6,54.2,False,False,False,False,True,False,False,False,False,True,False,False,False
3,21,5,29.1,77.4,68.0,68.3,63.6,False,True,False,False,False,False,False,False,False,False,False,False,True
4,16,5,30.9,68.2,65.0,49.1,53.9,False,False,False,False,False,False,True,False,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,3,9,26.8,82.5,179.0,40.3,44.6,False,False,False,False,True,False,False,False,True,False,False,False,False
7996,4,11,24.8,80.8,0.0,24.2,21.4,False,False,True,False,False,False,False,False,False,False,True,False,False
7997,5,11,25.2,86.4,0.0,21.7,24.3,True,False,False,False,False,False,False,False,True,False,False,False,False
7998,3,6,25.2,82.1,5.0,34.3,36.3,False,True,False,False,False,False,False,False,False,False,True,False,False


In [190]:
df_train.describe()

,hour,month,temperature,humidity,occupancy,previous_usage,energy_usage
count,8000.000000,8000.000000,7881.000000,7881.000000,7878.000000,7886.000000,8000.000000
mean,11.487750,6.542625,28.220886,78.051212,72.582127,49.674956,51.249212
std,6.913181,3.436373,1.876264,5.573045,74.333723,18.943682,18.448996
min,0.000000,1.000000,23.200000,61.100000,0.000000,5.300000,13.200000
25%,6.000000,4.000000,26.800000,74.300000,8.000000,35.900000,38.000000
50%,11.000000,7.000000,28.200000,78.000000,42.000000,46.400000,48.100000
75%,17.000000,10.000000,29.600000,81.700000,129.000000,61.400000,62.700000
max,23.000000,12.000000,35.800000,100.000000,337.000000,149.400000,143.900000


In [191]:
df_train.shape

(8000, 9)

In [192]:
df_train.isna().sum()

building_type       0
hour                0
day_of_week         0
month               0
temperature       119
humidity          119
occupancy         122
previous_usage    114
energy_usage        0
dtype: int64

In [193]:
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')

In [194]:
imputer

,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'mean'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False


In [195]:
day_to_int = {
    'Monday': 0,
    'Tuesday': 1,
    'Wednesday': 2,
    'Thursday': 3,
    'Friday': 4,
    'Saturday': 5,
    'Sunday': 6
}

# Add cyclical components to hour, day_of_week, and month features
def make_features_trig(df):
    df = df.copy()

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

    df["day_of_week"] = df["day_of_week"].map(day_to_int)
    df["day_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["day_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

    return df

# Impute any missing values in the dataframe and drop unnecessary columns, as well as one-hot-encode the building_type categorical variable
def impute_missing_values(df):
    df = df.copy()
    df.drop(columns=['hour', 'day_of_week', 'month'], inplace=True)
    df = pd.get_dummies(df, columns=['building_type'], drop_first=True)
    return pd.DataFrame(imputer.fit_transform(df), columns=df.columns)


def clean_df(df):
    return impute_missing_values(make_features_trig(df))
    


df_imputed = clean_df(df_train)
# feature_X_hot_trig = make_features_trig(df_train_hot)

In [196]:
# feature_df_trig.drop(columns=['hour', 'day_of_week', 'month'], inplace=True)
# feature_df_trig = pd.get_dummies(feature_df_trig, columns=['building_type'], drop_first=True)

In [197]:
df_imputed.isna().sum()


temperature                  0
humidity                     0
occupancy                    0
previous_usage               0
energy_usage                 0
hour_sin                     0
hour_cos                     0
day_sin                      0
day_cos                      0
month_sin                    0
month_cos                    0
is_weekend                   0
building_type_Business       0
building_type_Engineering    0
building_type_LectureHall    0
building_type_Library        0
building_type_Residential    0
building_type_Science        0
building_type_Sports         0
dtype: int64

In [198]:
df_imputed.head(10)

,temperature,humidity,occupancy,previous_usage,energy_usage,hour_sin,hour_cos,day_sin,day_cos,month_sin,month_cos,is_weekend,building_type_Business,building_type_Engineering,building_type_LectureHall,building_type_Library,building_type_Residential,building_type_Science,building_type_Sports
0,25.200000,81.9,31.0,28.0,27.2,0.500000,0.866025,0.000000,1.000000,-1.000000,-1.836970e-16,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,25.700000,83.7,14.0,72.6,67.2,0.258819,0.965926,0.974928,-0.222521,-0.500000,8.660254e-01,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,26.300000,84.7,199.0,51.6,54.2,0.965926,0.258819,-0.781831,0.623490,0.500000,8.660254e-01,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,29.100000,77.4,68.0,68.3,63.6,-0.707107,0.707107,0.974928,-0.222521,0.500000,-8.660254e-01,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,30.900000,68.2,65.0,49.1,53.9,-0.866025,-0.500000,0.974928,-0.222521,0.500000,-8.660254e-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
5,30.100000,70.7,17.0,79.3,81.3,-0.866025,-0.500000,-0.781831,0.623490,0.866025,5.000000e-01,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
6,26.000000,82.0,7.0,56.1,53.1,0.000000,1.000000,0.781831,0.623490,-1.000000,-1.836970e-16,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
7,28.220886,87.2,30.0,42.3,40.7,0.000000,1.000000,0.781831,0.623490,0.866025,5.000000e-01,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
8,29.200000,79.4,52.0,46.1,52.8,-0.258819,-0.965926,-0.781831,0.623490,0.500000,8.660254e-01,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
9,28.220886,63.8,27.0,27.3,29.7,0.258819,-0.965926,-0.781831,0.623490,1.000000,6.123234e-17,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [199]:
# X = feature_df_trig.copy()
# y = feature_df_trig['energy_usage']



In [200]:
X = df_imputed.copy().drop(columns=['energy_usage'])
y = df_imputed['energy_usage']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)

baseline_model = make_pipeline(StandardScaler(), LinearRegression())
baseline_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('linearregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](18,)","['temperature','humidity','occupancy',...,'building_type_Residential', 'building_type_Science','building_type_Sports']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,18
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [201]:
predictions = baseline_model.predict(X_val)
rmse = root_mean_squared_error(y_val, predictions)
print("Root Mean Squared Error (RMSE):", rmse)

Root Mean Squared Error (RMSE): 3.9587637845434056


Linear Regression

In [202]:

baseline_model = make_pipeline(StandardScaler(), LinearRegression())
baseline_model.fit(X_train, y_train)

linear_predictions = baseline_model.predict(X_val)
rmse = root_mean_squared_error(y_val, linear_predictions)
print("Root Mean Squared Error (RMSE):", rmse)


Root Mean Squared Error (RMSE): 3.9587637845434056


HistogramGradientBoost

In [203]:
hgb_baseline_model = make_pipeline(StandardScaler(),  HistGradientBoostingRegressor(
    max_iter=185,          # Maximum number of trees (iterations)
    learning_rate=0.1,     # Step size shrinkage used in updates
    max_leaf_nodes=31,     # Maximum number of leaves in each tree
    random_state=42
))
hgb_baseline_model.fit(X_train, y_train)

predictions = hgb_baseline_model.predict(X_val)
rmse = root_mean_squared_error(y_val, predictions)
print("Root Mean Squared Error (RMSE):", rmse)


Root Mean Squared Error (RMSE): 3.6038325514661067


Combined Model

In [204]:
predictions = baseline_model.predict(X_val)*0.4 + hgb_baseline_model.predict(X_val)*0.6
rmse = root_mean_squared_error(y_val, predictions)
print("Root Mean Squared Error (RMSE) for combined model:", rmse)

Root Mean Squared Error (RMSE) for combined model: 3.529590451936718


In [205]:
ridge_baseline_model = make_pipeline(StandardScaler(),  Ridge(
    alpha = 0.1
))
ridge_baseline_model.fit(X_train, y_train)

predictions = ridge_baseline_model.predict(X_val)
rmse = root_mean_squared_error(y_val, predictions)
print("Root Mean Squared Error (RMSE):", rmse)

Root Mean Squared Error (RMSE): 3.9587247388900635


In [206]:
predictions = baseline_model.predict(X_val)*0.4 + ridge_baseline_model.predict(X_val)*0.6
rmse = root_mean_squared_error(y_val, predictions)
print("Root Mean Squared Error (RMSE) for combined model:", rmse)

Root Mean Squared Error (RMSE) for combined model: 3.958740349798115


In [207]:
predictions = ridge_baseline_model.predict(X_val)*0.35 + hgb_baseline_model.predict(X_val)*0.65
rmse = root_mean_squared_error(y_val, predictions)
print("Root Mean Squared Error (RMSE) for combined model:", rmse)

Root Mean Squared Error (RMSE) for combined model: 3.5224011245659446


In [208]:
X_test_clean = clean_df(X_test_copy)

X_test_predictions = ridge_baseline_model.predict(X_test_clean)*0.35 + hgb_baseline_model.predict(X_test_clean)*0.65

print("Predictions for test set:", X_test_predictions)

Predictions for test set: [72.65119299 65.20304932 57.59125116 ... 16.67031901 38.66270773
 64.14632848]


In [209]:
X_test

,id,building_id,building_type,hour,day_of_week,month,temperature,humidity,occupancy,previous_usage
0,EC010541,SCI_B,Science,19,Monday,6,29.8,77.3,3.0,66.4
1,EC009755,ENG_A,Engineering,13,Sunday,7,29.9,70.6,88.0,63.5
2,EC008886,SCI_B,Science,1,Saturday,10,26.3,71.9,10.0,52.0
3,EC009317,SPT_A,Sports,11,Sunday,9,28.5,75.6,244.0,63.8
4,EC008958,LEC_A,LectureHall,2,Tuesday,2,25.8,83.2,0.0,11.5
...,...,...,...,...,...,...,...,...,...,...
2995,EC009548,LEC_A,LectureHall,16,Sunday,7,30.1,70.2,271.0,87.7
2996,EC010863,LEC_A,LectureHall,7,Saturday,8,27.5,78.4,0.0,24.1
2997,EC010682,ADM_A,Administration,3,Sunday,9,25.0,82.2,0.0,12.2
2998,EC010754,RES_B,Residential,11,Tuesday,2,29.4,NaN,NaN,37.0


In [210]:
submission_df = pd.DataFrame({
    'id': X_test['id'],         # Match the exact ID column name from test data
    'prediction': X_test_predictions          # Replace 'Prediction' with the target column name required
})

# 3. Save to a CSV file without the pandas row index
submission_df.to_csv('submission.csv', index=False)

## AI SLOP

In [218]:
from sklearn.linear_model import RidgeCV
from sklearn.experimental import enable_iterative_imputer  # must come before importing IterativeImputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor

train_full = pd.read_csv('train.csv')
test_full = pd.read_csv('test.csv')

num_cols = ['temperature', 'humidity', 'occupancy', 'previous_usage']
building_ids = sorted(train_full['building_id'].unique())
building_types = sorted(train_full['building_type'].unique())
day_to_int = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3,
              'Friday': 4, 'Saturday': 5, 'Sunday': 6}


def imputer_inputs(df):
    """The 4 columns that have blanks, plus never-blank columns the imputer can learn from."""
    X = df[num_cols].copy()
    X['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    X['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    X['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    X['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    X['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday']).astype(float)
    bid = pd.get_dummies(pd.Categorical(df['building_id'], categories=building_ids), prefix='bid').astype(float)
    bid.index = df.index
    return pd.concat([X, bid], axis=1)


def fit_imputer(train_df):
    """MissForest-style: each blank column is guessed by a tree model trained on the other columns."""
    imputer = IterativeImputer(
        estimator=ExtraTreesRegressor(n_estimators=100, min_samples_leaf=3, max_features=0.6,
                                      n_jobs=-1, random_state=0),
        max_iter=8, random_state=0)
    imputer.fit(imputer_inputs(train_df))
    return imputer


def fill_blanks(imputer, df):
    df = df.copy()
    df[num_cols] = imputer.transform(imputer_inputs(df))[:, :len(num_cols)]
    return df


def linear_features(df):
    X = df[num_cols].copy()
    X['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    X['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    X['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    X['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    X['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday']).astype(float)
    X['occ_x_weekend'] = X['occupancy'] * X['is_weekend']

    bid = pd.get_dummies(pd.Categorical(df['building_id'], categories=building_ids), prefix='bid').astype(float)
    btype = pd.get_dummies(pd.Categorical(df['building_type'], categories=building_types), prefix='bt').astype(float)
    bid.index = btype.index = df.index

    # each building gets its own slope for the main drivers
    inter = {}
    for col in ['occupancy', 'previous_usage', 'temperature', 'hour_sin', 'hour_cos']:
        for b in bid.columns:
            inter[f'{col}_x_{b}'] = X[col] * bid[b]
    for col in ['is_weekend', 'occ_x_weekend']:
        for b in btype.columns:
            inter[f'{col}_x_{b}'] = X[col] * btype[b]
    return pd.concat([X, btype, bid, pd.DataFrame(inter, index=df.index)], axis=1)


def tree_features(df):
    X = df[['hour', 'month'] + num_cols].copy()
    X['day'] = df['day_of_week'].map(day_to_int)
    X['is_weekend'] = (X['day'] >= 5).astype(int)
    X['building_type'] = pd.Categorical(df['building_type'], categories=building_types)
    X['building_id'] = pd.Categorical(df['building_id'], categories=building_ids)
    return X


def fit_and_predict(train_df, pred_df):
    imputer = fit_imputer(pd.concat([train_df, pred_df]))               # learns from TRAIN rows only
    train_df = fill_blanks(imputer, train_df)
    pred_df = fill_blanks(imputer, pred_df)
    y = train_df['energy_usage']

    ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 13)))
    ridge.fit(linear_features(train_df), y)
    ridge_pred = ridge.predict(linear_features(pred_df))

    # HGB predicts the change from previous_usage, then we add it back
    hgb = HistGradientBoostingRegressor(categorical_features='from_dtype', max_iter=1500, learning_rate=0.03,
                                        max_leaf_nodes=15, min_samples_leaf=30, l2_regularization=1.0,
                                        early_stopping=True, n_iter_no_change=50, random_state=42)
    hgb.fit(tree_features(train_df), y - train_df['previous_usage'])
    hgb_pred = hgb.predict(tree_features(pred_df)) + pred_df['previous_usage'].values

    return 0.8 * ridge_pred + 0.2 * hgb_pred

In [219]:
train_part, val_part = train_test_split(train_full, test_size=0.2, random_state=42)

val_pred = fit_and_predict(train_part, val_part)
print('Validation RMSE:', root_mean_squared_error(val_part['energy_usage'], val_pred))

rng = np.random.default_rng(0)
val_testlike = val_part.copy()
copied = test_full[num_cols].isna().values[rng.integers(0, len(test_full), len(val_part))]
val_testlike[num_cols] = val_testlike[num_cols].mask(copied)
val_pred = fit_and_predict(train_part, val_testlike)
print('Validation RMSE with test-like blanks:', root_mean_squared_error(val_part['energy_usage'], val_pred))

/Users/ethanis/Documents/Daddython/.venv/lib/python3.14/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Validation RMSE: 3.1167594067440456


/Users/ethanis/Documents/Daddython/.venv/lib/python3.14/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Validation RMSE with test-like blanks: 3.342479722783296


In [220]:
test_pred = fit_and_predict(train_full, test_full)
submission_df = pd.DataFrame({'id': test_full['id'], 'prediction': test_pred})
submission_df.to_csv('best_v2.csv', index=False)
submission_df.head()

/Users/ethanis/Documents/Daddython/.venv/lib/python3.14/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


,id,prediction
0,EC010541,71.625890
1,EC009755,66.400615
2,EC008886,56.184633
3,EC009317,67.147015
4,EC008958,20.930468


In [214]:
X_test.isna().sum()

id                  0
building_id         0
building_type       0
hour                0
day_of_week         0
month               0
temperature       211
humidity          183
occupancy         206
previous_usage    203
dtype: int64